# E5 — Width Distribution Comparison (All Windows)

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [ ]:
rows = []
for wk, wm in WINDOWS_META.items():
    if wk == 'W1':
        continue
    for sym in [wm['front'], wm['back']]:
        p1, p2, p3 = [], [], []
        for day in wm['days']:
            fpath = list(SEAGATE_DIR.glob(f'mbp10_*{sym}*{day}*.parquet'))
            if not fpath: continue
            df = pd.read_parquet(fpath[0], columns=['ts_event','bid_px_00','ask_px_00'])
            df['ts_event'] = pd.to_datetime(df['ts_event'])
            rth = ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) >= wm['rth_start_min']) &                   ((df['ts_event'].dt.hour*60+df['ts_event'].dt.minute) <= wm['rth_end_min'])
            df = df[rth]
            ba = (df['ask_px_00']-df['bid_px_00'])/TICK
            p1.append((ba==1).mean()); p2.append((ba==2).mean()); p3.append((ba>=3).mean())
            del df; gc.collect()
        rows.append(dict(window=wk, leg='front' if sym==wm['front'] else 'back',
                         pct_1t=np.mean(p1), pct_2t=np.mean(p2), pct_3t=np.mean(p3)))
df_wid = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, leg in zip(axes, ['front', 'back']):
    sub = df_wid[df_wid['leg']==leg]
    x = np.arange(len(sub))
    ax.bar(x, sub['pct_1t'], label='1-tick', color='#2ecc71', alpha=0.85)
    ax.bar(x, sub['pct_2t'], bottom=sub['pct_1t'].values, label='2-tick', color='#3498db', alpha=0.85)
    ax.bar(x, sub['pct_3t'], bottom=(sub['pct_1t']+sub['pct_2t']).values, label='3+tick', color='#e74c3c', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(sub['window'].values)
    ax.set_title(f'{leg.capitalize()} Contract BA Width')
    ax.set_ylabel('Avg fraction of quotes')
    ax.legend()
fig.suptitle('Bid-Ask Width Distribution Comparison — W2/W3/W4', fontsize=13)
save_fig(fig, 'E5_width_distribution_comparison.png')
